# 01 Best PseudoGT Learnability

            This notebook separates pseudo-GT learning from DQA aggregation.
            Every client starts from the copied 08_4 warmup checkpoint, then
            candidate pseudo-GT training profiles are compared on the same
            scene protocol used by the DQA experiments.


## Fixed condition

            - Start checkpoint: `checkpoints/round000_warmup.pt`
            - Output root: `output/`
            - Clients: `highway`, `citystreet`, `residential`
            - Evaluation: `highway`, `citystreet`, `residential`, `total`
            - Main question: can pseudo-GT training itself produce useful
              client updates before DQA-style class-wise aggregation is added?


In [ ]:
from pathlib import Path
            import os
            import subprocess
            import sys

            cwd = Path.cwd().resolve()
            if cwd.name == "notebooks":
                PROJECT_ROOT = cwd.parent
            elif (cwd / "pseudogt_learnability").exists():
                PROJECT_ROOT = cwd / "pseudogt_learnability"
            else:
                PROJECT_ROOT = cwd
            os.chdir(PROJECT_ROOT)
            print(PROJECT_ROOT)
            print((PROJECT_ROOT / "checkpoints" / "round000_warmup.pt").exists())


## Setup check

            This creates/reuses scene data lists in `output/` and verifies that
            the copied warmup checkpoint is visible from the new project.


In [ ]:
subprocess.run([
                sys.executable,
                "scripts/run_pseudogt_learnability_01.py",
                "--setup-only",
            ], check=True)


## Run 01

            The default run tests three profiles:

            - `backbone_obj_safe`
            - `neck_head_high_precision`
            - `all_consistency_lowlr`

            It saves client checkpoints, aggregate checkpoints, one-epoch
            server-repair checkpoints, and then evaluates them scene-wise.


In [ ]:
cmd = [
                sys.executable,
                "scripts/run_pseudogt_learnability_01.py",
                "--variants", "backbone_obj_safe,neck_head_high_precision,all_consistency_lowlr",
                "--batch-size", "160",
                "--workers", "8",
                "--gpus", "2",
                "--master-port", "30341",
                "--server-repair-epochs", "1",
                "--evaluate",
                "--classwise",
                "--no-eval-plots",
            ]
            print(" ".join(cmd))
            subprocess.run(cmd, check=True)


## Check generated checkpoints


In [ ]:
import pandas as pd

            checkpoint_table = PROJECT_ROOT / "output" / "stats" / "01_checkpoints.csv"
            if checkpoint_table.exists():
                ckpts = pd.read_csv(checkpoint_table)
                display(ckpts)
            else:
                print("No checkpoint table yet:", checkpoint_table)


## Evaluation summary

            The key metric is whether any pseudo-GT profile beats
            `warmup_global` on `total` mAP@0.5:0.95, or at least improves the
            matching client scene without damaging the total split.


In [ ]:
import pandas as pd

            summary_csv = PROJECT_ROOT / "output" / "validation_reports" / "paper_protocol_eval_summary.csv"
            if not summary_csv.exists():
                print("Evaluation summary is not available yet:", summary_csv)
            else:
                df = pd.read_csv(summary_csv)
                ok = df[df["status"].eq("ok")].copy()
                total = ok[ok["split"].isin(["total", "scene_total"])].copy()
                total = total.sort_values("map50_95", ascending=False)
                display(total[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])

                warm = total[total["checkpoint_label"].eq("warmup_global")]
                if not warm.empty:
                    base = float(warm.iloc[0]["map50_95"])
                    total["delta_map50_95_vs_warmup"] = total["map50_95"].astype(float) - base
                    display(total[["checkpoint_label", "map50", "map50_95", "delta_map50_95_vs_warmup"]])


## Scene-specific client learnability

            This view is useful even if the aggregate is weak.  If a client
            improves its own scene split, pseudo-GT may still be learnable, and
            the aggregation rule is the next target.  If no client improves its
            own split, the pseudo-GT training recipe itself is the bottleneck.


In [ ]:
if summary_csv.exists():
                ok = pd.read_csv(summary_csv)
                ok = ok[ok["status"].eq("ok")].copy()
                scene_rows = ok[ok["split"].isin(["highway", "citystreet", "residential"])].copy()
                display(scene_rows.sort_values(["split", "map50_95"], ascending=[True, False])[
                    ["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]
                ])


## Discord notification


In [ ]:
try:
                sys.path.insert(0, str(PROJECT_ROOT.parent))
                from notebook_notify import notify_discord

                msg = "PseudoGT Learnability 01 finished. Check output/validation_reports/paper_protocol_eval_summary.md"
                result = notify_discord(
                    msg,
                    title="PseudoGT Learnability 01",
                    context={"project": str(PROJECT_ROOT)},
                    fail_silently=True,
                )
                print(result)
            except Exception as exc:
                print("Discord notification skipped:", exc)
